# Laboratory 4 — NLP for Malicious SSH Log Analysis

**Objective:** Apply NLP techniques to cybersecurity by fine-tuning pre-trained language models for Named Entity Recognition (NER) on SSH session logs. The goal is to classify each "SSH word" (command, flag, parameter, separator) to its corresponding **MITRE tactic**.

**Core Activities:**
* Analyze datasets described by textual data
* Understand the impact of different tokenization strategies
* Experiment with various pre-trained Language Models and compare their performance
* Use fine-tuned models for inference to investigate cybersecurity threats

> ⚠️ **WARNING:** The dataset contains attacks collected from real deployments. **Do not carelessly execute scripts on your laptop.**
>
> ⚠️ **CRITICAL:** Do not decode base64 strings found in the logs. They may contain malicious scripts.

## Setup

In [1]:
# --- Check Python and pip versions ---
!python --version
!pip install --upgrade pip

Python 3.12.12
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 62.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [2]:
# --- Install required libraries ---
!pip install torch torchvision
!pip install numpy pandas scikit-learn matplotlib seaborn
!pip install transformers datasets
!pip install tqdm

In [3]:
# --- Import libraries ---
import os
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from copy import deepcopy
from itertools import chain

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoConfig,
    DataCollatorForTokenClassification,
    get_scheduler
)
from datasets import Dataset, DatasetDict

from tqdm.auto import tqdm

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

In [4]:
# --- Set device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


### Colab Pro

In [5]:
# --- Check GPU availability ---
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

/bin/bash: line 1: nvidia-smi: command not found


In [6]:
# --- Check RAM availability ---
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 13.6 gigabytes of available RAM

Not using a high-RAM runtime


### Paths setup


In [7]:
# --- Mount Google Drive (for Google Colab users) ---
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# --- Define Paths ---
group = 'AImSecure'
laboratory = 'Laboratory4'

base_path = '/content/drive/MyDrive/'
project_path = base_path + f'Projects/{group}/{laboratory}/'
data_path = project_path + 'data/'
results_path = project_path + 'results/'

# Ensure directories exist
os.makedirs(project_path, exist_ok=True)
os.makedirs(data_path, exist_ok=True)
os.makedirs(results_path, exist_ok=True)

print(f"Project path: {project_path}")
print(f"Data path: {data_path}")
print(f"Results path: {results_path}")

Project path: /content/drive/MyDrive/Projects/AImSecure/Laboratory4/
Data path: /content/drive/MyDrive/Projects/AImSecure/Laboratory4/data/
Results path: /content/drive/MyDrive/Projects/AImSecure/Laboratory4/results/


In [9]:
# --- Set visual style ---
sns.set(style="whitegrid", palette="muted", font_scale=1.1)

def save_plot(fig: plt.Figure, filename: str, path: str = "./plots/", fmt: str = "png", dpi: int = 300, close_fig: bool = False) -> None:
    """
    Save a Matplotlib figure in a specific to a specified directory.

    Args:
        fig (plt.Figure): Matplotlib figure object to save.
        filename (str): Name of the file to save (e.g., 'plot.png').
        path (str, optional): Directory path to save the figure. Defaults to './plots/'.
        fmt (str, optional): File format for the saved figure. Defaults to 'png'.
        dpi (int, optional): Dots per inch for the saved figure. Defaults to 300.

    Returns:
        None
    """
    # Ensure the directory exists
    os.makedirs(path, exist_ok=True)
    save_path = os.path.join(path, f"{filename}.{fmt}")

    # Save the figure
    fig.savefig(save_path, bbox_inches='tight', pad_inches=0.1, dpi=dpi, format=fmt)
    # plt.close(fig) # Removed to display plots in notebook

    if close_fig:
        plt.close(fig)

    print(f"Saved plot: {save_path}")

## Task 1 — Dataset Characterization

Before training, explore the data to understand distributions and tokenization.

In [10]:
# --- Create directory for plots ---
save_dir = results_path + 'images/' + 'task1_plots/'
os.makedirs(save_dir, exist_ok=True)

# --- Load the datasets ---
with open(data_path + 'train.json', 'r') as f:
    train_data = json.load(f)
with open(data_path + 'test.json', 'r') as f:
    test_data = json.load(f)

# Convert to DataFrames
train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)

print(f"Training dataset contains {len(train_df):,} sessions")
print(f"Testing dataset contains {len(test_df):,} sessions")
train_df.head()

Training dataset contains 251 sessions
Testing dataset contains 108 sessions


,session,label
0,rm -rf /var/run/1sh ; wget -c http://71.127.14...,"[Execution, Execution, Execution, Execution, E..."
1,cat /proc/cpuinfo | grep name | wc -l ; echo r...,"[Discovery, Discovery, Discovery, Discovery, D..."
2,cd /tmp || cd /var/run || cd /mnt || cd /root ...,"[Execution, Execution, Execution, Execution, E..."
3,ls -la /dev/ttyGSM* /var/spool/sms/* /var/log/...,"[Discovery, Discovery, Discovery, Discovery, D..."
4,cat /var/tmp/.systemcache436621 ; echo 1 > /va...,"[Discovery, Discovery, Discovery, Discovery, D..."


### 1.1 Label Analysis

**Instructions:**
- Count distinct tags and analyze distribution (bash words per tag).
- Create a Barplot showing tag distribution with 2 bars per tag: Train vs Test.

**Q:** What are the distinct MITRE tactic tags in the dataset? How are they distributed?

*Your analysis here*

In [ ]:
# --- Analyze tag distribution ---
# Extract all tags from train and test sets
train_tags = [tag for tags in train_df['label'] for tag in tags]
test_tags = [tag for tags in test_df['label'] for tag in tags]

# Count unique tags
unique_tags = list(set(train_tags + test_tags))
print(f"Distinct tags found: {len(unique_tags)}")
print(f"Tags: {unique_tags}")

# Create distribution counts
train_tag_counts = Counter(train_tags)
test_tag_counts = Counter(test_tags)

# Create DataFrame for plotting
tag_distribution = pd.DataFrame({
    'Tag': unique_tags,
    'Train': [train_tag_counts.get(tag, 0) for tag in unique_tags],
    'Test': [test_tag_counts.get(tag, 0) for tag in unique_tags]
}).sort_values('Train', ascending=False)

print("\nTag distribution:")
print(tag_distribution)

KeyError: 'tags'

In [ ]:
# --- Plot: Tag Distribution Barplot (Train vs Test) ---
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(tag_distribution))
width = 0.35

bars1 = ax.bar(x - width/2, tag_distribution['Train'], width, label='Train', color='#2E86C1', edgecolor='black')
bars2 = ax.bar(x + width/2, tag_distribution['Test'], width, label='Test', color='#27AE60', edgecolor='black')

ax.set_xlabel('MITRE Tactic Tag', fontsize=12)
ax.set_ylabel('Number of Bash Words', fontsize=12)
ax.set_title('Tag Distribution: Train vs Test', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(tag_distribution['Tag'], rotation=45, ha='right')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.7, axis='y')

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)

plt.tight_layout()
save_plot(fig, 'tag_distribution_barplot', path=save_dir)
plt.show()

### 1.2 Command Analysis: 'echo'

**Instructions:**
- Analyze how many different tags are assigned to the command `echo`.
- Find 1 example session where `echo` is tagged as 'Persistence' and 1 where it is 'Execution'.

**Q:** Why might the same command (`echo`) be labeled with different MITRE tactics in different contexts?

*Your analysis here*

In [ ]:
# --- Analyze the 'echo' command ---
# Find all occurrences of 'echo' and their associated tags
echo_tags = []
echo_examples = {'Persistence': None, 'Execution': None}

for idx, row in train_df.iterrows():
    # Split the sentence into words (use .split() not .split(" "))
    words = row['session'].split()
    tags = row['label']

    for word, tag in zip(words, tags):
        if word.lower() == 'echo':
            echo_tags.append(tag)
            # Store example sessions
            if tag == 'Persistence' and echo_examples['Persistence'] is None:
                echo_examples['Persistence'] = {
                    'session': row['session'],
                    'label': tags,
                    'index': idx
                }
            elif tag == 'Execution' and echo_examples['Execution'] is None:
                echo_examples['Execution'] = {
                    'session': row['session'],
                    'label': tags,
                    'index': idx
                }

# Count unique tags for 'echo'
echo_tag_counts = Counter(echo_tags)
print(f"Tags assigned to 'echo' command:")
for tag, count in echo_tag_counts.most_common():
    print(f"  {tag}: {count}")

In [ ]:
# --- Display example sessions for 'echo' with different tags ---
print("=" * 80)
print("Example session where 'echo' is tagged as 'Persistence':")
print("=" * 80)
if echo_examples['Persistence']:
    print(f"Session: {echo_examples['Persistence']['session']}")
    print(f"Labels: {echo_examples['Persistence']['label']}")
else:
    print("No 'Persistence' example found for 'echo'")

print("\n" + "=" * 80)
print("Example session where 'echo' is tagged as 'Execution':")
print("=" * 80)
if echo_examples['Execution']:
    print(f"Session: {echo_examples['Execution']['session']}")
    print(f"Labels: {echo_examples['Execution']['label']}")
else:
    print("No 'Execution' example found for 'echo'")

### 1.3 Session Length Analysis

**Instructions:**
- Calculate the number of Bash words per session.
- Plot an Estimated Cumulative Distribution Function (ECDF).

**Q:** What is the distribution of session lengths? Are there very long sessions that might cause issues during tokenization?

*Your analysis here*

In [ ]:
# --- Helper function for ECDF ---
def get_ecdf(list_numbers):
    """Calculate Empirical Cumulative Distribution Function."""
    counter = Counter(list_numbers)
    values = sorted(counter.keys())
    counts = [counter[value] for value in values]
    total = sum(counts)
    cumulative_probs = np.cumsum(counts) / total
    return values, cumulative_probs

In [ ]:
# --- Calculate session lengths (number of bash words per session) ---
train_session_lengths = [len(row['session'].split()) for _, row in train_df.iterrows()]
test_session_lengths = [len(row['session'].split()) for _, row in test_df.iterrows()]

print("Training set session length statistics:")
print(f"  - Min: {min(train_session_lengths)}")
print(f"  - Max: {max(train_session_lengths)}")
print(f"  - Mean: {np.mean(train_session_lengths):.2f}")
print(f"  - Median: {np.median(train_session_lengths):.2f}")

print("\nTest set session length statistics:")
print(f"  - Min: {min(test_session_lengths)}")
print(f"  - Max: {max(test_session_lengths)}")
print(f"  - Mean: {np.mean(test_session_lengths):.2f}")
print(f"  - Median: {np.median(test_session_lengths):.2f}")

In [ ]:
# --- Plot: ECDF of Session Lengths ---
all_session_lengths = train_session_lengths + test_session_lengths
lengths, cumulative_probs = get_ecdf(all_session_lengths)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))

for ax in axs:
    ax.step(lengths, cumulative_probs, where='post', color='royalblue', linewidth=2)
    ax.scatter(lengths, cumulative_probs, color='black', s=20, zorder=5)
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.set_ylabel('ECDF', fontsize=12)
    ax.set_ylim(0, 1.05)

axs[0].set_xlabel('Number of Bash Words per Session', fontsize=12)
axs[0].set_title('ECDF of Session Length (Linear Scale)', fontsize=12)

axs[1].set_xlabel('Number of Bash Words per Session (log scale)', fontsize=12)
axs[1].set_title('ECDF of Session Length (Log Scale)', fontsize=12)
axs[1].set_xscale('log')

fig.suptitle('Empirical Cumulative Distribution Function of Session Lengths', fontsize=14, y=1.02)
plt.tight_layout()
save_plot(fig, 'session_length_ecdf', path=save_dir)
plt.show()

## Task 2 — Tokenization

Compare two specific pre-trained models:
1. **BERT-base** (`bert-base-uncased`): Pre-trained on English text
2. **UnixCoder-base** (`microsoft/unixcoder-base`): Pre-trained on natural language and source code (Python, Java, Bash)

In [ ]:
# --- Create directory for plots ---
save_dir = results_path + 'images/' + 'task2_plots/'
os.makedirs(save_dir, exist_ok=True)

# --- Load tokenizers ---
BERT_MODEL = "bert-base-uncased"
UNIXCODER_MODEL = "microsoft/unixcoder-base"

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL, add_prefix_space=True)
unixcoder_tokenizer = AutoTokenizer.from_pretrained(UNIXCODER_MODEL, add_prefix_space=True)

print(f"BERT vocabulary size: {bert_tokenizer.vocab_size:,}")
print(f"UnixCoder vocabulary size: {unixcoder_tokenizer.vocab_size:,}")
print(f"\nModel max length (context size):")
print(f"  BERT: {bert_tokenizer.model_max_length:,}")
print(f"  UnixCoder: {unixcoder_tokenizer.model_max_length:,}")

### 2.1 Micro-test: Tokenization Comparison

**Instructions:**
- Tokenize the list `['cat', 'shell', 'echo', 'top', 'chpasswd', 'crontab', 'wget', 'busybox', 'grep']`
- Compare how each tokenizer divides commands.

**Q:** Which tokenizer has a better token-to-word ratio for these SSH commands?

*Your analysis here*

In [ ]:
# --- Micro-test: Tokenize specific SSH commands ---
test_commands = ['cat', 'shell', 'echo', 'top', 'chpasswd', 'crontab', 'wget', 'busybox', 'grep']

print("Tokenization comparison for SSH commands:")
print("=" * 70)
print(f"{'Command':<12} | {'BERT Tokens':<25} | {'UnixCoder Tokens':<25}")
print("-" * 70)

bert_total_tokens = 0
unix_total_tokens = 0

for cmd in test_commands:
    # Tokenize with BERT
    bert_tokens = bert_tokenizer.tokenize(cmd)
    # Tokenize with UnixCoder
    unix_tokens = unixcoder_tokenizer.tokenize(cmd)

    bert_total_tokens += len(bert_tokens)
    unix_total_tokens += len(unix_tokens)

    print(f"{cmd:<12} | {str(bert_tokens):<25} | {str(unix_tokens):<25}")

print("-" * 70)
print(f"\nTotal tokens for {len(test_commands)} commands:")
print(f"  BERT: {bert_total_tokens} tokens (ratio: {bert_total_tokens/len(test_commands):.2f})")
print(f"  UnixCoder: {unix_total_tokens} tokens (ratio: {unix_total_tokens/len(test_commands):.2f})")

### 2.2 Corpus Tokenization

**Instructions:**
- Tokenize the entire training corpus with both tokenizers.
- Compare average tokens per session and max tokens per session.

**Q:** Why might one tokenizer produce more tokens than another? (Hint: Consider usage of `[UNK]` token)

*Your analysis here*

In [ ]:
# --- Tokenize the entire training corpus ---
train_sentences = train_df['session'].tolist()

# Tokenize with BERT
bert_tokenized = bert_tokenizer(train_sentences)
bert_tokens_per_session = [len(ids) for ids in bert_tokenized['input_ids']]

# Tokenize with UnixCoder
unix_tokenized = unixcoder_tokenizer(train_sentences)
unix_tokens_per_session = [len(ids) for ids in unix_tokenized['input_ids']]

print("Corpus Tokenization Statistics:")
print("=" * 60)
print(f"{'Metric':<30} | {'BERT':<12} | {'UnixCoder':<12}")
print("-" * 60)
print(f"{'Average tokens/session':<30} | {np.mean(bert_tokens_per_session):<12.2f} | {np.mean(unix_tokens_per_session):<12.2f}")
print(f"{'Max tokens/session':<30} | {max(bert_tokens_per_session):<12} | {max(unix_tokens_per_session):<12}")
print(f"{'Min tokens/session':<30} | {min(bert_tokens_per_session):<12} | {min(unix_tokens_per_session):<12}")
print(f"{'Median tokens/session':<30} | {np.median(bert_tokens_per_session):<12.1f} | {np.median(unix_tokens_per_session):<12.1f}")

In [ ]:
# --- Analyze [UNK] token usage ---
bert_unk_id = bert_tokenizer.unk_token_id
unix_unk_id = unixcoder_tokenizer.unk_token_id

bert_unk_count = sum(1 for ids in bert_tokenized['input_ids'] for id in ids if id == bert_unk_id)
unix_unk_count = sum(1 for ids in unix_tokenized['input_ids'] for id in ids if id == unix_unk_id)

print(f"\n[UNK] Token Analysis:")
print(f"  BERT [UNK] occurrences: {bert_unk_count}")
print(f"  UnixCoder [UNK] occurrences: {unix_unk_count}")

### 2.3 Truncation Analysis

**Instructions:**
- Set `max_length=512`. Count how many sessions are currently truncated.
- Analyze the session with the maximum number of tokens.

**Q:** Why is the longest session so long? What does it contain?

*Your analysis here*

In [ ]:
# --- Truncation analysis with max_length=512 ---
MAX_LENGTH = 512

# Count sessions exceeding max_length
bert_truncated = sum(1 for n in bert_tokens_per_session if n > MAX_LENGTH)
unix_truncated = sum(1 for n in unix_tokens_per_session if n > MAX_LENGTH)

print(f"Truncation Analysis (max_length={MAX_LENGTH}):")
print(f"  BERT - Sessions truncated: {bert_truncated}/{len(train_df)} ({bert_truncated/len(train_df)*100:.2f}%)")
print(f"  UnixCoder - Sessions truncated: {unix_truncated}/{len(train_df)} ({unix_truncated/len(train_df)*100:.2f}%)")

In [ ]:
# --- Analyze the longest session ---
max_bert_idx = np.argmax(bert_tokens_per_session)
max_unix_idx = np.argmax(unix_tokens_per_session)

print(f"Longest session (by BERT tokenization):")
print(f"  Index: {max_bert_idx}")
print(f"  BERT tokens: {bert_tokens_per_session[max_bert_idx]}")
print(f"  UnixCoder tokens: {unix_tokens_per_session[max_bert_idx]}")
print(f"  Number of words: {len(train_df.iloc[max_bert_idx]['session'].split())}")
print(f"\n  Session preview (first 500 chars):")
print(f"  {train_df.iloc[max_bert_idx]['session'][:500]}...")

# Look at individual word lengths
max_session_words = train_df.iloc[max_bert_idx]['session'].split()
word_lengths = [len(w) for w in max_session_words]
print(f"\n  Word length statistics in this session:")
print(f"    Max word length: {max(word_lengths)}")
print(f"    Words > 30 chars: {sum(1 for l in word_lengths if l > 30)}")

### 2.4 Optimization: Word Truncation

**Instructions:**
- Truncate individual words longer than **30 characters**.
- Re-tokenize and plot "Number of words" vs "Number of tokens" for both models.
- Report which tokenizer has the best ratio after optimization.
- Report how many sessions are still truncated.

**Q:** How does truncating long words affect the tokenization efficiency?

*Your analysis here*

In [ ]:
# --- Function to truncate long words ---
MAX_WORD_LENGTH = 30

def truncate_long_words(sentence, max_len=MAX_WORD_LENGTH):
    """Truncate words longer than max_len characters."""
    words = sentence.split()
    truncated_words = [w[:max_len] if len(w) > max_len else w for w in words]
    return ' '.join(truncated_words)

# Apply truncation to training data
train_sentences_truncated = [truncate_long_words(s) for s in train_sentences]

# Example of truncation effect
print("Example of word truncation:")
long_word_example = "aGVsbG8gd29ybGQgdGhpcyBpcyBhIHZlcnkgbG9uZyBiYXNlNjQgc3RyaW5n"
print(f"  Original: {long_word_example}")
print(f"  Truncated: {long_word_example[:MAX_WORD_LENGTH]}")

In [ ]:
# --- Re-tokenize with truncated words ---
bert_tokenized_trunc = bert_tokenizer(train_sentences_truncated)
unix_tokenized_trunc = unixcoder_tokenizer(train_sentences_truncated)

bert_tokens_trunc = [len(ids) for ids in bert_tokenized_trunc['input_ids']]
unix_tokens_trunc = [len(ids) for ids in unix_tokenized_trunc['input_ids']]

# Calculate words per session
words_per_session = [len(s.split()) for s in train_sentences_truncated]

print("After word truncation (max 30 chars):")
print("=" * 60)
print(f"{'Metric':<30} | {'BERT':<12} | {'UnixCoder':<12}")
print("-" * 60)
print(f"{'Average tokens/session':<30} | {np.mean(bert_tokens_trunc):<12.2f} | {np.mean(unix_tokens_trunc):<12.2f}")
print(f"{'Max tokens/session':<30} | {max(bert_tokens_trunc):<12} | {max(unix_tokens_trunc):<12}")

# Count truncated sessions
bert_still_truncated = sum(1 for n in bert_tokens_trunc if n > MAX_LENGTH)
unix_still_truncated = sum(1 for n in unix_tokens_trunc if n > MAX_LENGTH)
print(f"\n{'Sessions still truncated':<30} | {bert_still_truncated:<12} | {unix_still_truncated:<12}")

In [ ]:
# --- Plot: Number of Words vs Number of Tokens ---
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

# BERT plot
axs[0].scatter(words_per_session, bert_tokens_trunc, alpha=0.6,
               c='#2E86C1', edgecolors='black', linewidth=0.5, s=50)
axs[0].plot([0, max(words_per_session)], [0, max(words_per_session)],
            'r--', label='1:1 ratio', linewidth=2)
axs[0].axhline(y=512, color='firebrick', linestyle='-', linewidth=2, label='Max length (512)')
axs[0].set_xlabel('Number of Words', fontsize=12)
axs[0].set_ylabel('Number of Tokens', fontsize=12)
axs[0].set_title('BERT Tokenization', fontsize=14)
axs[0].legend()
axs[0].grid(True, linestyle='--', alpha=0.7)

# Calculate and display ratio
bert_ratio = np.mean(np.array(bert_tokens_trunc) / np.array(words_per_session))
axs[0].text(0.05, 0.95, f'Avg ratio: {bert_ratio:.2f}', transform=axs[0].transAxes,
            fontsize=11, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# UnixCoder plot
axs[1].scatter(words_per_session, unix_tokens_trunc, alpha=0.6,
               c='#27AE60', edgecolors='black', linewidth=0.5, s=50)
axs[1].plot([0, max(words_per_session)], [0, max(words_per_session)],
            'r--', label='1:1 ratio', linewidth=2)
axs[1].axhline(y=512, color='firebrick', linestyle='-', linewidth=2, label='Max length (512)')
axs[1].set_xlabel('Number of Words', fontsize=12)
axs[1].set_ylabel('Number of Tokens', fontsize=12)
axs[1].set_title('UnixCoder Tokenization', fontsize=14)
axs[1].legend()
axs[1].grid(True, linestyle='--', alpha=0.7)

# Calculate and display ratio
unix_ratio = np.mean(np.array(unix_tokens_trunc) / np.array(words_per_session))
axs[1].text(0.05, 0.95, f'Avg ratio: {unix_ratio:.2f}', transform=axs[1].transAxes,
            fontsize=11, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.suptitle('Words vs Tokens (After Word Truncation)', fontsize=14, y=1.02)
plt.tight_layout()
save_plot(fig, 'words_vs_tokens', path=save_dir)
plt.show()

print(f"\nBest token-to-word ratio: {'UnixCoder' if unix_ratio < bert_ratio else 'BERT'} ({min(bert_ratio, unix_ratio):.2f})")

## Task 3 — Model Training

Fine-tune models for Token Classification (classifying each token into a MITRE Tactic).

In [ ]:
# --- Create directory for plots ---
save_dir = results_path + 'images/' + 'task3_plots/'
os.makedirs(save_dir, exist_ok=True)

# --- Prepare datasets with word truncation ---
def prepare_dataset(df, max_word_len=30):
    """Prepare dataset with truncated words."""
    data = []
    for _, row in df.iterrows():
        words = row['session'].split()
        tags = row['label']
        # Truncate long words
        truncated_words = [w[:max_word_len] if len(w) > max_word_len else w for w in words]
        data.append({
            'session': ' '.join(truncated_words),
            'words': truncated_words,
            'label': tags
        })
    return pd.DataFrame(data)

train_df_prepared = prepare_dataset(train_df)
test_df_prepared = prepare_dataset(test_df)

# Create validation split
train_df_final, val_df_final = train_test_split(train_df_prepared, test_size=0.2, random_state=42)

print(f"Training set: {len(train_df_final)} sessions")
print(f"Validation set: {len(val_df_final)} sessions")
print(f"Test set: {len(test_df_prepared)} sessions")

### 3.0 Data Preparation for NER

Prepare the datasets for token classification using HuggingFace utilities.

In [ ]:
# --- Create label mappings ---
unique_labels = list(set(tag for tags in train_df['label'] for tag in tags))
unique_labels = sorted(unique_labels)

id2label = {i: label for i, label in enumerate(unique_labels)}
label2id = {label: i for i, label in enumerate(unique_labels)}

print(f"Number of unique labels: {len(unique_labels)}")
print(f"Labels: {unique_labels}")
print(f"\nLabel to ID mapping: {label2id}")

In [ ]:
# --- Convert DataFrames to HuggingFace Datasets ---
def convert_to_hf_dataset(df):
    """Convert pandas DataFrame to HuggingFace Dataset."""
    return Dataset.from_pandas(df.reset_index(drop=True))

full_ds = DatasetDict({
    "train": convert_to_hf_dataset(train_df_final),
    "valid": convert_to_hf_dataset(val_df_final),
    "test": convert_to_hf_dataset(test_df_prepared)
})

print("Dataset structure:")
print(full_ds)

In [ ]:
# --- Convert tags to IDs ---
def convert_labels_to_ids(sample):
    """Convert tag strings to numeric IDs."""
    sample['label_id'] = [label2id[tag] for tag in sample['label']]
    return sample

encoded_dataset = full_ds.map(convert_labels_to_ids)
print("Example:")
print(f"  Labels: {encoded_dataset['train'][0]['label']}")
print(f"  Label IDs: {encoded_dataset['train'][0]['label_id']}")

In [ ]:
# --- Tokenization and Label Alignment Functions ---
def align_labels_with_tokens(labels, word_ids):
    """
    Align token-level labels with word-level labels for tokenized text.

    Args:
        labels: Original word-level labels
        word_ids: List of word indices that each token corresponds to

    Returns:
        List of token-aligned labels where:
        - Special tokens (None word_ids) are assigned -100
        - First token of each word gets the word's label
        - Continuation tokens also get the word's label
    """
    new_labels = []
    current_word = None
    for word_id in word_ids:
        if word_id != current_word:
            current_word = word_id
            label = -100 if word_id is None else labels[word_id]
            new_labels.append(label)
        elif word_id is None:
            new_labels.append(-100)
        else:
            label = labels[word_id]
            new_labels.append(label)
    return new_labels

def create_tokenize_and_align_function(tokenizer):
    """Create a tokenization function for a specific tokenizer."""
    def tokenize_and_align_labels(samples):
        split_sentences = [sentence.split() for sentence in samples["session"]]
        tokenized_inputs = tokenizer(
            split_sentences,
            truncation=True,
            max_length=512,
            is_split_into_words=True
        )
        all_tags = samples["label_id"]
        new_labels = []
        for i, tags in enumerate(all_tags):
            word_ids = tokenized_inputs.word_ids(i)
            aligned_labels = align_labels_with_tokens(tags, word_ids)
            new_labels.append(aligned_labels)
        tokenized_inputs["labels"] = new_labels
        return tokenized_inputs
    return tokenize_and_align_labels

In [ ]:
# --- Helper functions for training and evaluation ---
def postprocess(predictions, labels, id2label):
    """Post-process model predictions and labels for evaluation."""
    predictions = predictions.detach().cpu().clone().numpy()
    labels = labels.detach().cpu().clone().numpy()
    # Remove ignored index (special tokens) and convert to labels
    true_labels = [[id2label[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    return true_predictions, true_labels

def compute_metrics(full_predictions, full_labels):
    """Calculate token-level metrics for token classification."""
    flat_predictions = list(chain(*full_predictions))
    flat_labels = list(chain(*full_labels))

    token_accuracy = accuracy_score(flat_labels, flat_predictions)
    token_precision = precision_score(flat_labels, flat_predictions, average='macro', zero_division=0)
    token_recall = recall_score(flat_labels, flat_predictions, average='macro', zero_division=0)
    token_f1 = f1_score(flat_labels, flat_predictions, average='macro', zero_division=0)

    return {
        "token_accuracy": token_accuracy,
        "token_precision": token_precision,
        "token_recall": token_recall,
        "token_f1": token_f1,
    }

def compute_per_class_f1(full_predictions, full_labels, unique_labels):
    """Calculate per-class F1 scores."""
    flat_predictions = list(chain(*full_predictions))
    flat_labels = list(chain(*full_labels))

    per_class_f1 = f1_score(flat_labels, flat_predictions, labels=unique_labels, average=None, zero_division=0)
    return dict(zip(unique_labels, per_class_f1))

def compute_session_fidelity(full_predictions, full_labels):
    """
    Calculate average session fidelity.
    Fidelity = Correct Predictions / Total Tokens per session
    """
    fidelities = []
    for pred, label in zip(full_predictions, full_labels):
        if len(label) > 0:
            correct = sum(1 for p, l in zip(pred, label) if p == l)
            fidelity = correct / len(label)
            fidelities.append(fidelity)
    return np.mean(fidelities) if fidelities else 0.0

In [ ]:
# --- Plotting helper functions ---
def plot_training_stats(title, training_losses, validation_losses=None, best_epoch=None, save_path=None):
    """Plot training and validation losses."""
    plt.style.use('seaborn-v0_8-paper')
    colors = {
        'training': '#2E86C1',
        'validation': '#27AE60',
        'best_epoch': '#E74C3C',
    }

    if validation_losses is not None:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    else:
        fig, ax1 = plt.subplots(1, figsize=(5, 4))

    ax1.plot(training_losses, color=colors['training'], linewidth=2)
    ax1.set_title('Training Loss', fontsize=12, pad=10)
    ax1.set_ylabel('Loss', fontsize=10)
    ax1.set_xlabel('Epoch', fontsize=10)

    if validation_losses is not None:
        if best_epoch is not None:
            ax1.axvline(x=best_epoch, color=colors['best_epoch'], linestyle='--', alpha=0.8, label='Best Epoch')
        ax2.plot(validation_losses, color=colors['validation'], linewidth=2)
        if best_epoch is not None:
            ax2.axvline(x=best_epoch, color=colors['best_epoch'], linestyle='--', alpha=0.8, label='Best Epoch')
        ax2.set_title('Validation Loss', fontsize=12, pad=10)
        ax2.set_ylabel('Loss', fontsize=10)
        ax2.set_xlabel('Epoch', fontsize=10)
        axs = [ax1, ax2]
    else:
        axs = [ax1]

    for ax in axs:
        ax.grid(True, linestyle='--', alpha=0.7)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        if validation_losses is not None:
            ax.legend(fontsize=8)

    fig.suptitle(f'{title} - Training Progress', fontsize=14, y=1.02)
    plt.tight_layout()

    if save_path:
        save_plot(fig, f'{title.lower().replace(" ", "_")}_losses', path=save_path)
    plt.show()

def plot_per_class_f1(per_class_f1, title, save_path=None):
    """Plot per-class F1 scores as a barplot."""
    fig, ax = plt.subplots(figsize=(10, 5))

    labels = list(per_class_f1.keys())
    values = list(per_class_f1.values())

    bars = ax.bar(labels, values, color='#2E86C1', edgecolor='black', linewidth=1)

    # Add value labels
    for bar, val in zip(bars, values):
        ax.annotate(f'{val:.3f}', xy=(bar.get_x() + bar.get_width()/2, val),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9)

    ax.set_xlabel('MITRE Tactic', fontsize=12)
    ax.set_ylabel('F1 Score', fontsize=12)
    ax.set_title(f'{title} - Per-Class F1 Score', fontsize=14)
    ax.set_ylim(0, 1.1)
    ax.grid(True, linestyle='--', alpha=0.7, axis='y')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()

    if save_path:
        save_plot(fig, f'{title.lower().replace(" ", "_")}_per_class_f1', path=save_path)
    plt.show()

def plot_confusion_matrix(full_predictions, full_labels, id2label, title, normalize=False, save_path=None):
    """Plot confusion matrix."""
    flat_predictions = list(chain(*full_predictions))
    flat_labels = list(chain(*full_labels))

    labels_list = list(id2label.values())
    cm = confusion_matrix(flat_labels, flat_predictions, labels=labels_list)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
        fmt = '.2f'
    else:
        fmt = 'd'

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=labels_list, yticklabels=labels_list, ax=ax)
    ax.set_ylabel('True label', fontsize=12)
    ax.set_xlabel('Predicted label', fontsize=12)
    ax.set_title(f'{title} - {"Normalized " if normalize else ""}Confusion Matrix', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()

    if save_path:
        suffix = '_normalized' if normalize else ''
        save_plot(fig, f'{title.lower().replace(" ", "_")}_confusion_matrix{suffix}', path=save_path)
    plt.show()

In [ ]:
# --- Training function ---
def train_model(model, train_dataloader, eval_dataloader, optimizer, lr_scheduler,
                n_epochs, device, id2label, model_name="Model"):
    """
    Train a token classification model.

    Returns:
        best_model: Model with best validation loss
        train_losses: List of training losses per epoch
        val_losses: List of validation losses per epoch
        best_epoch: Epoch with best validation loss
    """
    model = model.to(device)

    best_val_loss = np.inf
    best_epoch = 0
    best_model = deepcopy(model)

    train_losses = []
    val_losses = []

    num_training_steps = n_epochs * len(train_dataloader)
    progress_bar = tqdm(range(num_training_steps), desc=f"Training {model_name}")

    for epoch in range(n_epochs):
        # Training
        model.train()
        train_loss = 0
        for batch in train_dataloader:
            batch = {key: value.to(device) for key, value in batch.items()}
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"]
            )
            loss = outputs.loss
            train_loss += loss.detach().cpu().clone().numpy()
            loss.backward()
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            progress_bar.update(1)

        train_losses.append(train_loss / len(train_dataloader))

        # Validation
        model.eval()
        full_predictions, full_labels = [], []
        val_loss = 0

        for batch in eval_dataloader:
            batch = {key: value.to(device) for key, value in batch.items()}
            with torch.no_grad():
                outputs = model(**batch)
            val_loss += outputs.loss.detach().cpu().clone().numpy()
            predictions = outputs.logits.argmax(dim=-1)
            labels = batch["labels"]
            true_predictions, true_labels = postprocess(predictions, labels, id2label)
            full_predictions += true_predictions
            full_labels += true_labels

        val_loss = val_loss / len(eval_dataloader)
        val_losses.append(val_loss)

        metrics = compute_metrics(full_predictions, full_labels)

        print(f"Epoch {epoch}: acc={metrics['token_accuracy']:.4f}, f1={metrics['token_f1']:.4f}, val_loss={val_loss:.4f}")

        if val_loss <= best_val_loss:
            best_epoch = epoch
            best_val_loss = val_loss
            best_model = deepcopy(model)

    return best_model, train_losses, val_losses, best_epoch

In [ ]:
# --- Evaluation function ---
def evaluate_model(model, test_dataloader, device, id2label, unique_labels, model_name="Model"):
    """
    Evaluate a trained model on test data.

    Returns:
        Dictionary containing all metrics
    """
    model.eval()
    full_predictions, full_labels = [], []

    for batch in tqdm(test_dataloader, desc=f"Evaluating {model_name}"):
        batch = {key: value.to(device) for key, value in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        predictions = outputs.logits.argmax(dim=-1)
        labels = batch["labels"]
        true_predictions, true_labels = postprocess(predictions, labels, id2label)
        full_predictions += true_predictions
        full_labels += true_labels

    # Compute metrics
    metrics = compute_metrics(full_predictions, full_labels)
    per_class_f1 = compute_per_class_f1(full_predictions, full_labels, unique_labels)
    session_fidelity = compute_session_fidelity(full_predictions, full_labels)

    print(f"\n{model_name} Test Results:")
    print("=" * 50)
    print(f"Token Accuracy:  {metrics['token_accuracy']*100:6.2f}%")
    print(f"Token Precision: {metrics['token_precision']*100:6.2f}%")
    print(f"Token Recall:    {metrics['token_recall']*100:6.2f}%")
    print(f"Token F1 (Macro): {metrics['token_f1']*100:6.2f}%")
    print(f"Session Fidelity: {session_fidelity*100:6.2f}%")

    return {
        'metrics': metrics,
        'per_class_f1': per_class_f1,
        'session_fidelity': session_fidelity,
        'predictions': full_predictions,
        'labels': full_labels
    }

### 3.A BERT Fine-tuning (Pre-trained)

**Instructions:**
- Load `bert-base-uncased` with pre-trained weights.
- Fine-tune for token classification (NER).
- Report metrics: Token accuracy, macro precision/recall/F1, per-class F1 barplot, and Session Fidelity.

**Q:** Can the model achieve good results with only ~251 training samples?

*Your analysis here*

In [ ]:
# --- Prepare BERT tokenizer and data ---
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL, add_prefix_space=True)

# Create tokenization function for BERT
tokenize_bert = create_tokenize_and_align_function(bert_tokenizer)

# Tokenize datasets
original_columns = encoded_dataset["train"].column_names
bert_tokenized_datasets = encoded_dataset.map(
    tokenize_bert,
    batched=True,
    remove_columns=original_columns,
)

print("BERT Tokenized Dataset:")
print(bert_tokenized_datasets)

In [ ]:
# --- Create DataLoaders for BERT ---
bert_data_collator = DataCollatorForTokenClassification(tokenizer=bert_tokenizer)

BATCH_SIZE = 16

bert_train_dataloader = DataLoader(
    bert_tokenized_datasets["train"],
    shuffle=True,
    collate_fn=bert_data_collator,
    batch_size=BATCH_SIZE,
)
bert_eval_dataloader = DataLoader(
    bert_tokenized_datasets["valid"],
    collate_fn=bert_data_collator,
    batch_size=BATCH_SIZE,
)
bert_test_dataloader = DataLoader(
    bert_tokenized_datasets["test"],
    collate_fn=bert_data_collator,
    batch_size=BATCH_SIZE,
)

print(f"Training batches: {len(bert_train_dataloader)}")
print(f"Validation batches: {len(bert_eval_dataloader)}")
print(f"Test batches: {len(bert_test_dataloader)}")

In [ ]:
# --- Load pre-trained BERT model ---
bert_pretrained_model = AutoModelForTokenClassification.from_pretrained(
    BERT_MODEL,
    id2label=id2label,
    label2id=label2id,
)

# Count parameters
total_params = sum(p.numel() for p in bert_pretrained_model.parameters())
trainable_params = sum(p.numel() for p in bert_pretrained_model.parameters() if p.requires_grad)
print(f"BERT Pre-trained Model:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

In [ ]:
# --- Training configuration for BERT ---
BERT_LR = 5e-5
BERT_EPOCHS = 20

bert_optimizer = AdamW(bert_pretrained_model.parameters(), lr=BERT_LR)

num_training_steps = BERT_EPOCHS * len(bert_train_dataloader)
bert_lr_scheduler = get_scheduler(
    "linear",
    optimizer=bert_optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

print(f"Training configuration:")
print(f"  Learning rate: {BERT_LR}")
print(f"  Epochs: {BERT_EPOCHS}")
print(f"  Total training steps: {num_training_steps}")

In [ ]:
# --- Train BERT model ---
bert_best_model, bert_train_losses, bert_val_losses, bert_best_epoch = train_model(
    model=bert_pretrained_model,
    train_dataloader=bert_train_dataloader,
    eval_dataloader=bert_eval_dataloader,
    optimizer=bert_optimizer,
    lr_scheduler=bert_lr_scheduler,
    n_epochs=BERT_EPOCHS,
    device=device,
    id2label=id2label,
    model_name="BERT Pretrained"
)

print(f"\nBest epoch: {bert_best_epoch}")

In [ ]:
# --- Plot BERT training progress ---
plot_training_stats("BERT Pretrained", bert_train_losses, bert_val_losses, bert_best_epoch, save_path=save_dir)

In [ ]:
# --- Evaluate BERT model on test set ---
bert_results = evaluate_model(
    model=bert_best_model,
    test_dataloader=bert_test_dataloader,
    device=device,
    id2label=id2label,
    unique_labels=unique_labels,
    model_name="BERT Pretrained"
)

In [ ]:
# --- Plot per-class F1 scores for BERT ---
plot_per_class_f1(bert_results['per_class_f1'], "BERT Pretrained", save_path=save_dir)

In [ ]:
# --- Plot confusion matrices for BERT ---
plot_confusion_matrix(bert_results['predictions'], bert_results['labels'], id2label,
                      "BERT Pretrained", normalize=False, save_path=save_dir)
plot_confusion_matrix(bert_results['predictions'], bert_results['labels'], id2label,
                      "BERT Pretrained", normalize=True, save_path=save_dir)

### 3.B Baseline: "Naked" BERT (No Pre-training)

**Instructions:**
- Load BERT architecture *without* pre-trained weights (random initialization).
- Train end-to-end and compare performance against the pre-trained version.

**Q:** How much does pre-training help compared to training from scratch?

*Your analysis here*

In [ ]:
# --- Load BERT model WITHOUT pre-trained weights (random initialization) ---
# Get the configuration
bert_config = AutoConfig.from_pretrained(
    BERT_MODEL,
    id2label=id2label,
    label2id=label2id,
)

# Initialize model with random weights using the configuration
bert_naked_model = AutoModelForTokenClassification.from_config(bert_config)

total_params = sum(p.numel() for p in bert_naked_model.parameters())
print(f"'Naked' BERT Model (random initialization):")
print(f"  Total parameters: {total_params:,}")

In [ ]:
# --- Training configuration for Naked BERT ---
NAKED_BERT_LR = 5e-5
NAKED_BERT_EPOCHS = 20

naked_bert_optimizer = AdamW(bert_naked_model.parameters(), lr=NAKED_BERT_LR)

num_training_steps = NAKED_BERT_EPOCHS * len(bert_train_dataloader)
naked_bert_lr_scheduler = get_scheduler(
    "linear",
    optimizer=naked_bert_optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

In [ ]:
# --- Train Naked BERT model ---
naked_bert_best_model, naked_bert_train_losses, naked_bert_val_losses, naked_bert_best_epoch = train_model(
    model=bert_naked_model,
    train_dataloader=bert_train_dataloader,
    eval_dataloader=bert_eval_dataloader,
    optimizer=naked_bert_optimizer,
    lr_scheduler=naked_bert_lr_scheduler,
    n_epochs=NAKED_BERT_EPOCHS,
    device=device,
    id2label=id2label,
    model_name="Naked BERT"
)

print(f"\nBest epoch: {naked_bert_best_epoch}")

In [ ]:
# --- Plot Naked BERT training progress ---
plot_training_stats("Naked BERT", naked_bert_train_losses, naked_bert_val_losses,
                    naked_bert_best_epoch, save_path=save_dir)

In [ ]:
# --- Evaluate Naked BERT model on test set ---
naked_bert_results = evaluate_model(
    model=naked_bert_best_model,
    test_dataloader=bert_test_dataloader,
    device=device,
    id2label=id2label,
    unique_labels=unique_labels,
    model_name="Naked BERT"
)

In [ ]:
# --- Plot per-class F1 scores for Naked BERT ---
plot_per_class_f1(naked_bert_results['per_class_f1'], "Naked BERT", save_path=save_dir)

### 3.C UnixCoder Fine-tuning

**Instructions:**
- Fine-tune `microsoft/unixcoder-base`.
- Compare performance with BERT.

**Q:** Does the code-specific pre-training yield better results on SSH logs?

*Your analysis here*

In [ ]:
# --- Prepare UnixCoder tokenizer and data ---
unixcoder_tokenizer = AutoTokenizer.from_pretrained(UNIXCODER_MODEL, add_prefix_space=True)

# Create tokenization function for UnixCoder
tokenize_unixcoder = create_tokenize_and_align_function(unixcoder_tokenizer)

# Tokenize datasets
unix_tokenized_datasets = encoded_dataset.map(
    tokenize_unixcoder,
    batched=True,
    remove_columns=original_columns,
)

print("UnixCoder Tokenized Dataset:")
print(unix_tokenized_datasets)

In [ ]:
# --- Create DataLoaders for UnixCoder ---
unix_data_collator = DataCollatorForTokenClassification(tokenizer=unixcoder_tokenizer)

unix_train_dataloader = DataLoader(
    unix_tokenized_datasets["train"],
    shuffle=True,
    collate_fn=unix_data_collator,
    batch_size=BATCH_SIZE,
)
unix_eval_dataloader = DataLoader(
    unix_tokenized_datasets["valid"],
    collate_fn=unix_data_collator,
    batch_size=BATCH_SIZE,
)
unix_test_dataloader = DataLoader(
    unix_tokenized_datasets["test"],
    collate_fn=unix_data_collator,
    batch_size=BATCH_SIZE,
)

In [ ]:
# --- Load pre-trained UnixCoder model ---
unixcoder_model = AutoModelForTokenClassification.from_pretrained(
    UNIXCODER_MODEL,
    id2label=id2label,
    label2id=label2id,
)

total_params = sum(p.numel() for p in unixcoder_model.parameters())
print(f"UnixCoder Model:")
print(f"  Total parameters: {total_params:,}")

In [ ]:
# --- Training configuration for UnixCoder ---
UNIX_LR = 5e-5
UNIX_EPOCHS = 20

unix_optimizer = AdamW(unixcoder_model.parameters(), lr=UNIX_LR)

num_training_steps = UNIX_EPOCHS * len(unix_train_dataloader)
unix_lr_scheduler = get_scheduler(
    "linear",
    optimizer=unix_optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

In [ ]:
# --- Train UnixCoder model ---
unix_best_model, unix_train_losses, unix_val_losses, unix_best_epoch = train_model(
    model=unixcoder_model,
    train_dataloader=unix_train_dataloader,
    eval_dataloader=unix_eval_dataloader,
    optimizer=unix_optimizer,
    lr_scheduler=unix_lr_scheduler,
    n_epochs=UNIX_EPOCHS,
    device=device,
    id2label=id2label,
    model_name="UnixCoder"
)

print(f"\nBest epoch: {unix_best_epoch}")

In [ ]:
# --- Plot UnixCoder training progress ---
plot_training_stats("UnixCoder", unix_train_losses, unix_val_losses, unix_best_epoch, save_path=save_dir)

In [ ]:
# --- Evaluate UnixCoder model on test set ---
unix_results = evaluate_model(
    model=unix_best_model,
    test_dataloader=unix_test_dataloader,
    device=device,
    id2label=id2label,
    unique_labels=unique_labels,
    model_name="UnixCoder"
)

In [ ]:
# --- Plot per-class F1 scores for UnixCoder ---
plot_per_class_f1(unix_results['per_class_f1'], "UnixCoder", save_path=save_dir)

### 3.D Frozen Layer Experiment

**Instructions:**
- Take the best model from previous steps.
- Fine-tune *only* the last 2 encoding layers + classification head (freeze the rest).
- Compare training speed, parameter count, and performance loss vs. full fine-tuning.

**Q:** How does freezing layers affect training efficiency and final performance?

*Your analysis here*

In [ ]:
# --- Load fresh BERT model for frozen layer experiment ---
bert_frozen_model = AutoModelForTokenClassification.from_pretrained(
    BERT_MODEL,
    id2label=id2label,
    label2id=label2id,
)

# Freeze all layers in base model
for param in bert_frozen_model.base_model.parameters():
    param.requires_grad = False

# Unfreeze the last 2 encoder layers
for param in bert_frozen_model.base_model.encoder.layer[-2:].parameters():
    param.requires_grad = True

# Classification head is always trainable
for param in bert_frozen_model.classifier.parameters():
    param.requires_grad = True

# Count parameters
total_params = sum(p.numel() for p in bert_frozen_model.parameters())
trainable_params = sum(p.numel() for p in bert_frozen_model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"BERT Frozen Model:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Frozen parameters: {frozen_params:,}")
print(f"  Trainable ratio: {trainable_params/total_params*100:.2f}%")

In [ ]:
# --- Show which layers are trainable ---
print("Trainable layers:")
for name, param in bert_frozen_model.named_parameters():
    if param.requires_grad:
        print(f"  {name}")

In [ ]:
# --- Training configuration for Frozen BERT ---
# Can use higher LR since we're training fewer parameters
FROZEN_LR = 5e-5
FROZEN_EPOCHS = 20

frozen_optimizer = AdamW(filter(lambda p: p.requires_grad, bert_frozen_model.parameters()), lr=FROZEN_LR)

num_training_steps = FROZEN_EPOCHS * len(bert_train_dataloader)
frozen_lr_scheduler = get_scheduler(
    "linear",
    optimizer=frozen_optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

In [ ]:
# --- Train Frozen BERT model ---
import time

start_time = time.time()
frozen_best_model, frozen_train_losses, frozen_val_losses, frozen_best_epoch = train_model(
    model=bert_frozen_model,
    train_dataloader=bert_train_dataloader,
    eval_dataloader=bert_eval_dataloader,
    optimizer=frozen_optimizer,
    lr_scheduler=frozen_lr_scheduler,
    n_epochs=FROZEN_EPOCHS,
    device=device,
    id2label=id2label,
    model_name="BERT Frozen"
)
frozen_training_time = time.time() - start_time

print(f"\nBest epoch: {frozen_best_epoch}")
print(f"Training time: {frozen_training_time:.2f} seconds")

In [ ]:
# --- Plot Frozen BERT training progress ---
plot_training_stats("BERT Frozen", frozen_train_losses, frozen_val_losses, frozen_best_epoch, save_path=save_dir)

In [ ]:
# --- Evaluate Frozen BERT model on test set ---
frozen_results = evaluate_model(
    model=frozen_best_model,
    test_dataloader=bert_test_dataloader,
    device=device,
    id2label=id2label,
    unique_labels=unique_labels,
    model_name="BERT Frozen"
)

In [ ]:
# --- Compare all models ---
print("\n" + "=" * 80)
print("MODEL COMPARISON SUMMARY")
print("=" * 80)

comparison_data = {
    'Model': ['BERT Pretrained', 'Naked BERT', 'UnixCoder', 'BERT Frozen'],
    'Accuracy': [
        bert_results['metrics']['token_accuracy']*100,
        naked_bert_results['metrics']['token_accuracy']*100,
        unix_results['metrics']['token_accuracy']*100,
        frozen_results['metrics']['token_accuracy']*100,
    ],
    'F1 (Macro)': [
        bert_results['metrics']['token_f1']*100,
        naked_bert_results['metrics']['token_f1']*100,
        unix_results['metrics']['token_f1']*100,
        frozen_results['metrics']['token_f1']*100,
    ],
    'Session Fidelity': [
        bert_results['session_fidelity']*100,
        naked_bert_results['session_fidelity']*100,
        unix_results['session_fidelity']*100,
        frozen_results['session_fidelity']*100,
    ],
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

In [ ]:
# --- Plot comparison barplot ---
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

models = comparison_df['Model']
x = np.arange(len(models))

# Accuracy
axs[0].bar(x, comparison_df['Accuracy'], color=['#2E86C1', '#E74C3C', '#27AE60', '#9B59B6'], edgecolor='black')
axs[0].set_ylabel('Accuracy (%)', fontsize=12)
axs[0].set_title('Token Accuracy', fontsize=14)
axs[0].set_xticks(x)
axs[0].set_xticklabels(models, rotation=45, ha='right')
axs[0].set_ylim(0, 100)
axs[0].grid(True, linestyle='--', alpha=0.7, axis='y')

# F1 Score
axs[1].bar(x, comparison_df['F1 (Macro)'], color=['#2E86C1', '#E74C3C', '#27AE60', '#9B59B6'], edgecolor='black')
axs[1].set_ylabel('F1 Score (%)', fontsize=12)
axs[1].set_title('Macro F1 Score', fontsize=14)
axs[1].set_xticks(x)
axs[1].set_xticklabels(models, rotation=45, ha='right')
axs[1].set_ylim(0, 100)
axs[1].grid(True, linestyle='--', alpha=0.7, axis='y')

# Session Fidelity
axs[2].bar(x, comparison_df['Session Fidelity'], color=['#2E86C1', '#E74C3C', '#27AE60', '#9B59B6'], edgecolor='black')
axs[2].set_ylabel('Fidelity (%)', fontsize=12)
axs[2].set_title('Session Fidelity', fontsize=14)
axs[2].set_xticks(x)
axs[2].set_xticklabels(models, rotation=45, ha='right')
axs[2].set_ylim(0, 100)
axs[2].grid(True, linestyle='--', alpha=0.7, axis='y')

fig.suptitle('Model Comparison', fontsize=16, y=1.02)
plt.tight_layout()
save_plot(fig, 'model_comparison', path=save_dir)
plt.show()

## Task 4 — Inference

Use the best fine-tuned model (with word truncation applied) on `cyberlab.csv` for inference.

**Key Constraints:**
- **Word-Level Tagging:** Use the prediction of the **first token** of a word to tag the entire word.
- **Truncated Sessions:** Only consider Bash words that received a prediction.

In [ ]:
# --- Create directory for plots ---
save_dir = results_path + 'images/' + 'task4_plots/'
os.makedirs(save_dir, exist_ok=True)

# --- Load inference dataset ---
cyberlab_df = pd.read_csv(data_path + 'cyberlab.csv')
print(f"Cyberlab dataset contains {len(cyberlab_df):,} sessions")
cyberlab_df.head()

### 4.1 Inference with Best Model

Apply the best model to predict MITRE tactics for each word in the cyberlab sessions.

**Note:** We use the first token's prediction to tag the entire word.

In [ ]:
# --- Select best model for inference ---
# Determine best model based on F1 score
all_models = {
    'BERT Pretrained': (bert_best_model, bert_tokenizer, bert_results),
    'UnixCoder': (unix_best_model, unixcoder_tokenizer, unix_results),
    'BERT Frozen': (frozen_best_model, bert_tokenizer, frozen_results),
}

best_model_name = max(all_models.keys(),
                      key=lambda x: all_models[x][2]['metrics']['token_f1'])
best_model, best_tokenizer, _ = all_models[best_model_name]

print(f"Selected best model for inference: {best_model_name}")

In [ ]:
# --- Inference function with word-level prediction ---
def predict_session(model, tokenizer, session_text, device, id2label, max_word_len=30):
    """
    Predict MITRE tactics for each word in a session.
    Uses the first token's prediction for each word.

    Args:
        model: Fine-tuned model
        tokenizer: Corresponding tokenizer
        session_text: Raw session text
        device: CPU or GPU
        id2label: Mapping from label IDs to label names
        max_word_len: Maximum word length (words are truncated)

    Returns:
        List of (word, predicted_tag) tuples
    """
    model.eval()

    # Split and truncate words
    words = session_text.split()
    truncated_words = [w[:max_word_len] if len(w) > max_word_len else w for w in words]

    if not truncated_words:
        return []

    # Tokenize
    tokenized = tokenizer(
        truncated_words,
        is_split_into_words=True,
        truncation=True,
        max_length=512,
        return_tensors='pt'
    )

    # Move to device
    tokenized = {k: v.to(device) for k, v in tokenized.items()}

    # Predict
    with torch.no_grad():
        outputs = model(**tokenized)

    predictions = outputs.logits.argmax(dim=-1)[0].cpu().numpy()
    word_ids = tokenized.word_ids(batch_index=0)

    # Get first token prediction for each word
    word_predictions = []
    seen_words = set()

    for idx, word_id in enumerate(word_ids):
        if word_id is not None and word_id not in seen_words:
            if word_id < len(truncated_words):  # Only include non-truncated words
                pred_label = id2label[predictions[idx]]
                word_predictions.append((truncated_words[word_id], pred_label))
                seen_words.add(word_id)

    return word_predictions

In [ ]:
# --- Run inference on cyberlab dataset ---
# Assuming cyberlab_df has a 'session' or 'command' column - adjust based on actual column name
# Check the actual column names
print("Cyberlab columns:", cyberlab_df.columns.tolist())

# Identify the session/command column (adjust as needed)
session_column = None
for col in ['session', 'command', 'sentences', 'input', 'text']:
    if col in cyberlab_df.columns:
        session_column = col
        break

# If no standard column found, use the first column
if session_column is None:
    session_column = cyberlab_df.columns[0]

print(f"Using column '{session_column}' for sessions")

In [ ]:
# --- Run inference on all sessions ---
inference_results = []

best_model.eval()
best_model = best_model.to(device)

for idx, row in tqdm(cyberlab_df.iterrows(), total=len(cyberlab_df), desc="Running inference"):
    session_text = str(row[session_column])

    # Get predictions
    predictions = predict_session(best_model, best_tokenizer, session_text, device, id2label)

    # Extract fingerprint (sequence of predicted tactics)
    fingerprint = tuple([pred[1] for pred in predictions])

    # Store result
    inference_results.append({
        'index': idx,
        'session': session_text,
        'words': [p[0] for p in predictions],
        'predictions': [p[1] for p in predictions],
        'fingerprint': fingerprint,
        'date': row.get('date', row.get('timestamp', None))  # Try to get date column
    })

print(f"\nInference completed for {len(inference_results):,} sessions")

### 4.2 Command Profiling

**Instructions:**
- Focus on: `cat`, `grep`, `echo`, `rm`
- Report tag frequency for each command in a table.
- Provide 1 example session for each unique tuple (command, predicted tag).

**Q:** What tactics are most commonly associated with these commands?

*Your analysis here*

In [ ]:
# --- Command Profiling: Analyze specific commands ---
target_commands = ['cat', 'grep', 'echo', 'rm']

# Collect command-tag pairs
command_tag_data = {cmd: Counter() for cmd in target_commands}
command_tag_examples = {cmd: {} for cmd in target_commands}

for result in inference_results:
    for word, tag in zip(result['words'], result['predictions']):
        word_lower = word.lower()
        if word_lower in target_commands:
            command_tag_data[word_lower][tag] += 1
            # Store one example for each (command, tag) pair
            if tag not in command_tag_examples[word_lower]:
                command_tag_examples[word_lower][tag] = result['session'][:200]  # Truncate for display

# Display tag frequency table
print("Command-Tag Frequency Table:")
print("=" * 70)
for cmd in target_commands:
    print(f"\n{cmd.upper()}:")
    for tag, count in command_tag_data[cmd].most_common():
        print(f"  {tag:<20}: {count:>6}")

In [ ]:
# --- Display example sessions for each (command, tag) pair ---
print("\nExample Sessions for Each (Command, Tag) Pair:")
print("=" * 80)

for cmd in target_commands:
    print(f"\n{cmd.upper()} Examples:")
    print("-" * 80)
    for tag, example in command_tag_examples[cmd].items():
        print(f"\n  Tag: {tag}")
        print(f"  Session: {example}...")

### 4.3 Fingerprinting & Campaign Detection

**Instructions:**
- Define fingerprint as the sequence of predicted MITRE Tactics for a session.
- Identify unique fingerprints in the inference set.
- Sort fingerprints by "date of birth" (first appearance).
- Assign an ID (0 to N) to fingerprints based on this sort order.
- Create a scatter plot (Time on X-axis vs. Fingerprint ID on Y-axis).

**Q:** Are there fingerprints that appear suddenly in large numbers, suggesting attack campaigns?

*Your analysis here*

In [ ]:
# --- Create inference results DataFrame ---
inference_df = pd.DataFrame(inference_results)

# Convert date column to datetime if it exists
date_column = None
for col in ['date', 'timestamp', 'time', 'datetime']:
    if col in cyberlab_df.columns:
        date_column = col
        break

if date_column:
    inference_df['date'] = pd.to_datetime(cyberlab_df[date_column])
else:
    # Create a synthetic date column if none exists (based on index)
    print("Warning: No date column found. Using session index as time proxy.")
    inference_df['date'] = pd.date_range(start='2019-05-01', periods=len(inference_df), freq='H')

# Convert fingerprint to string for easier handling
inference_df['fingerprint_str'] = inference_df['fingerprint'].apply(lambda x: '|'.join(x) if x else 'EMPTY')

print(f"Total sessions: {len(inference_df):,}")
print(f"Unique fingerprints: {inference_df['fingerprint_str'].nunique():,}")

In [ ]:
# --- Assign fingerprint IDs based on first appearance (date of birth) ---
# Find the first occurrence date for each fingerprint
fingerprint_first_seen = inference_df.groupby('fingerprint_str')['date'].min().reset_index()
fingerprint_first_seen.columns = ['fingerprint_str', 'first_seen']

# Sort by first seen date and assign IDs
fingerprint_first_seen = fingerprint_first_seen.sort_values('first_seen').reset_index(drop=True)
fingerprint_first_seen['fingerprint_id'] = fingerprint_first_seen.index

# Merge back to main dataframe
inference_df = inference_df.merge(fingerprint_first_seen[['fingerprint_str', 'fingerprint_id']], on='fingerprint_str')

print(f"Fingerprint IDs assigned (0 to {fingerprint_first_seen['fingerprint_id'].max()})")
print(f"\nTop 10 most common fingerprints:")
print(inference_df['fingerprint_str'].value_counts().head(10))

In [ ]:
# --- Create scatter plot: Time vs Fingerprint ID ---
# Aggregate by day and fingerprint for point sizing
inference_df['date_day'] = inference_df['date'].dt.date

# Count sessions per fingerprint per day
daily_counts = inference_df.groupby(['date_day', 'fingerprint_id']).size().reset_index(name='count')
daily_counts['date_day'] = pd.to_datetime(daily_counts['date_day'])

# Create the scatter plot (similar to Figure 2 in the PDF)
fig, ax = plt.subplots(figsize=(14, 8))

# Use scatter with size based on count
scatter = ax.scatter(
    daily_counts['date_day'],
    daily_counts['fingerprint_id'],
    s=daily_counts['count'] * 5,  # Scale point size by count
    c=daily_counts['count'],      # Color by count
    cmap='YlOrRd',
    alpha=0.6,
    edgecolors='black',
    linewidth=0.5
)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Sessions per Day', fontsize=12)

ax.set_xlabel('Time', fontsize=12)
ax.set_ylabel('Fingerprint ID (ordered by first appearance)', fontsize=12)
ax.set_title('Attack Campaign Detection: Session Fingerprints Over Time', fontsize=14)
ax.grid(True, linestyle='--', alpha=0.5)

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
save_plot(fig, 'fingerprint_scatter_plot', path=save_dir)
plt.show()

In [ ]:
# --- Identify potential attack campaigns ---
# Look for fingerprints with sudden large appearances
campaign_threshold = 50  # Fingerprints appearing more than 50 times in a day

potential_campaigns = daily_counts[daily_counts['count'] >= campaign_threshold].copy()
potential_campaigns = potential_campaigns.merge(
    fingerprint_first_seen[['fingerprint_id', 'fingerprint_str']],
    on='fingerprint_id'
)

print(f"Potential Attack Campaigns (≥{campaign_threshold} sessions/day):")
print("=" * 80)
if len(potential_campaigns) > 0:
    for _, row in potential_campaigns.sort_values('count', ascending=False).head(20).iterrows():
        # Get example session for this fingerprint
        example = inference_df[inference_df['fingerprint_id'] == row['fingerprint_id']].iloc[0]
        print(f"\nDate: {row['date_day'].strftime('%Y-%m-%d')}")
        print(f"Fingerprint ID: {row['fingerprint_id']}")
        print(f"Sessions: {row['count']}")
        print(f"Fingerprint: {row['fingerprint_str'][:80]}...")
        print(f"Example session: {example['session'][:100]}...")
else:
    print("No significant campaign spikes detected with current threshold.")

In [ ]:
# --- Plot fingerprint frequency distribution ---
fingerprint_counts = inference_df['fingerprint_str'].value_counts()

fig, axs = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of fingerprint frequencies
axs[0].hist(fingerprint_counts.values, bins=50, color='#2E86C1', edgecolor='black', alpha=0.7)
axs[0].set_xlabel('Session Count', fontsize=12)
axs[0].set_ylabel('Number of Fingerprints', fontsize=12)
axs[0].set_title('Distribution of Fingerprint Frequencies', fontsize=14)
axs[0].set_yscale('log')
axs[0].grid(True, linestyle='--', alpha=0.7)

# Top 20 most common fingerprints
top_fingerprints = fingerprint_counts.head(20)
axs[1].barh(range(len(top_fingerprints)), top_fingerprints.values, color='#27AE60', edgecolor='black')
axs[1].set_yticks(range(len(top_fingerprints)))
axs[1].set_yticklabels([f[:30] + '...' if len(f) > 30 else f for f in top_fingerprints.index], fontsize=8)
axs[1].set_xlabel('Session Count', fontsize=12)
axs[1].set_title('Top 20 Most Common Fingerprints', fontsize=14)
axs[1].grid(True, linestyle='--', alpha=0.7, axis='x')
axs[1].invert_yaxis()

plt.tight_layout()
save_plot(fig, 'fingerprint_frequency_distribution', path=save_dir)
plt.show()

In [ ]:
# --- Summary Statistics ---
print("\nInference Summary Statistics:")
print("=" * 60)
print(f"Total sessions analyzed: {len(inference_df):,}")
print(f"Unique fingerprints: {inference_df['fingerprint_str'].nunique():,}")
print(f"Date range: {inference_df['date'].min()} to {inference_df['date'].max()}")
print(f"\nMost common fingerprint appears in {fingerprint_counts.iloc[0]:,} sessions")
print(f"Singleton fingerprints: {(fingerprint_counts == 1).sum():,}")

---

## Conclusion

*Your final analysis and conclusions here*

**Summary:**
- Task 1: Characterized the SSH session dataset, analyzing tag distributions and session lengths
- Task 2: Compared BERT and UnixCoder tokenization strategies, optimized with word truncation
- Task 3: Fine-tuned multiple models (BERT, Naked BERT, UnixCoder, Frozen BERT) for NER
- Task 4: Applied the best model for inference on cyberlab data, detecting attack fingerprints

**Key Findings:**
- *Add your observations about model performance*
- *Add your observations about tokenization efficiency*
- *Add your observations about detected attack campaigns*